# Automated Wavelength Solution

[Chapter 10](10-wavelength-solution-interactive.ipynb) needed a person to pick six lines against a
reference figure, which is fine at a terminal and impossible in a pipeline reducing 300 frames
overnight. A pipeline has something the interactive session did not: **prior knowledge of the
solution**. The solution depends on the instrument setup (grism, slit, detector), and on a stable,
well-characterized instrument it changes little from night to night. Once arcs from a
configuration have been calibrated and checked, the wavelength at the reference pixel, the
dispersion there and the curvature are known to within a small range.

`fit_dispersion` searches inside that range. Instead of being given correct pairs, it looks for the
polynomial coefficients that put the observed lines closest to catalog lines, using differential
evolution over a bounded parameter space and a KD-tree to find each line's nearest catalog
neighbor, and it returns the same `WavelengthSolution1D`. What it needs are the **bounds**: ranges
for the wavelength and the dispersion at the reference pixel, and limits on the higher-order
coefficients, all bracketing the prior. This chapter derives them from a validated solution, fits,
and checks the result.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract
from specreduce.tracing import FitTrace, FlatTrace
from specreduce.tilt_solution import TiltSolution
from specreduce.wavecal1d import WavelengthCalibration1D
from specreduce.wavesol1d import WavelengthSolution1D

from common import (
    GTC_HGAR,
    TILT_SOLUTION_FILE,
    WAVELENGTH_SOLUTION_FILE,
    fit_tilt_solution,
    fit_wavelength_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]
pixels = np.arange(float(nx))

## Setup

The inputs are Chapter 10's: arc spectra extracted from the rectified frames, the same line lists,
and `find_lines` for detection. Every fit below builds a fresh `WavelengthCalibration1D`, so that no
run inherits another's matches.

In [ ]:
reference_row = FlatTrace(arcs[0], 135)

if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified_arcs = [solution.resample(frame) for frame in arcs]
rectified_obj = solution.resample(obj)


spectra = [
    BoxcarExtract(frame, reference_row, width=15).spectrum for frame in rectified_arcs
]

## The prior

In a pipeline the prior is a configuration entry per instrument setup: the wavelength at the
reference pixel, the dispersion there and the higher-order terms, each with a range drawn from a
set of checked calibrations of that setup, whose spread measures how much the instrument drifts.
This chapter has one validated solution, the one Chapter 10 wrote, so it stands in for the set and
the ranges are set by judgment; the guard refits it if the file is missing. The solution is a
polynomial in `pixel - ref_pixel`, so its coefficients are directly the quantities
`fit_dispersion` asks about.

In [ ]:
if not WAVELENGTH_SOLUTION_FILE.exists():  # normally written by Chapter 10
    fit_wavelength_solution(arcs, solution).to_asdf(WAVELENGTH_SOLUTION_FILE)

prior = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)
prior_axis = prior.pix_to_wav(pixels)
c0, c1, c2, c3 = prior.p2w[1].parameters

print("prior (Chapter 10's solution)")
print(f"  wavelength at pixel 512:  {c0:.2f} A")
print(f"  dispersion there:         {c1:.4f} A/pix")
print(f"  quadratic, cubic terms:   {c2:.3e}, {c3:.3e}")
print(f"  coverage:                 {prior_axis[0]:.0f} to {prior_axis[-1]:.0f} A")

The bounds follow from those numbers, with widths comfortable for this instrument. The
higher-order limits come from the prior as well, at three times its coefficients.

In [ ]:
config = dict(
    wavelength_bounds=(c0 - 150, c0 + 150),
    dispersion_bounds=(c1 - 0.2, c1 + 0.2),
    higher_order_limits=[3 * abs(c2), 3 * abs(c3)],
    degree=3,
)

## The automated fit

The global search is stochastic. `fit_dispersion` takes a `seed`, handed to
`differential_evolution`, so that the same seed gives the same solution and a pipeline run is
repeatable. After the fit we print the numbers a pipeline can compute for itself, the rms, the
matched line count and the coverage, and one it cannot: the largest deviation from the prior,
available here only because the prior is a validated solution for this very arc.

In [ ]:
calibration = WavelengthCalibration1D(
    arc_spectra=spectra,
    line_lists=[GTC_HGAR, ["NeI"], ["XeI"]],
    line_list_bounds=(5000, 10500),
    ref_pixel=512,
    unit=u.angstrom,
    wave_air=True,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # unconverged refits on non-line detections
    calibration.find_lines(fwhm=2.5, noise_factor=15)
    calibration.fit_dispersion(seed=0, **config)

total_detected = sum(len(lines) for lines in calibration.observed_lines)

matched = sum(
    np.ma.asarray(lines).compressed().size
    for lines in calibration.observed_line_locations
)
# Freeze the matched set before reading the rms, so that the number describes
# the lines counted above.
calibration.remove_unmatched_lines()
axis = calibration.solution.pix_to_wav(pixels)
deviation = np.max(np.abs(axis - prior_axis))

print(
    f"rms {calibration.rms('wavelength'):.2f} A, {matched} of {total_detected} lines matched, "
    f"coverage {axis[0]:.0f} to {axis[-1]:.0f} A"
)
print(f"largest deviation from the prior: {deviation:.2f} A ({deviation / c1:.2f} pix)")

The fit lands within a third of a pixel of the prior, from bounds alone and without a single
identified line.

How tight the bounds must be is worth knowing, because a search box that is too wide fails without
warning. With the bounds above, ten different seeds all land within a pixel of the prior. Widen any
of them and convergence becomes a matter of the seed: opening the wavelength and dispersion bounds
to ±1000 Å and ±1 Å per pixel loses two seeds in five, loosening the higher-order limits to ten
times the prior's coefficients also loses two, and at a hundred times three fail. A failed search
lands hundreds to thousands of Ångström away, having laid the detections smoothly onto the wrong
catalog lines. The bounds are where the prior enters the fit, so keep them as tight as the spread
of validated solutions allows. Degree 3 and the default population size of 30 suffice on these
arcs; a higher degree adds a coefficient to the search and buys nothing.

## A pipeline-shaped function

`calibrate` puts the pieces together: the class's own line detection, the bounds from the validated
solution, a seed, and a final `refine_fit` at the interactive chapter's 2 Å. A real pipeline adds
one thing this book does not, an automated check that the fit converged, since a search that lands
on a wrong solution still reports a plausible rms over the lines it matched and nobody is looking
at the plot. The matched line count and the fitted coverage are the natural inputs to such a
check.

In [ ]:
def calibrate(arc_spectra, line_lists, config, seed=0):
    """Fit a wavelength solution automatically from prior-derived bounds."""
    calibration = WavelengthCalibration1D(
        arc_spectra=arc_spectra,
        line_lists=line_lists,
        line_list_bounds=(5000, 10500),
        ref_pixel=512,
        unit=u.angstrom,
        wave_air=True,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        calibration.find_lines(fwhm=2.5, noise_factor=15)
        calibration.fit_dispersion(seed=seed, **config)
        calibration.refine_fit(max_match_distance=2.0)
    calibration.remove_unmatched_lines()
    return calibration


result = calibrate(spectra, [GTC_HGAR, ["NeI"], ["XeI"]], config)
axis = result.solution.pix_to_wav(pixels)
deviation = np.max(np.abs(axis - prior_axis))

matched = sum(len(lines) for lines in result.observed_lines)
print(
    f"rms {result.rms('wavelength'):.3f} A over {matched} lines, "
    f"coverage {axis[0]:.0f}-{axis[-1]:.0f} A"
)
print(
    f"agreement with the interactive solution: {deviation:.2f} A ({deviation / c1:.2f} pix)"
)

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 4.0), constrained_layout=True, height_ratios=[2, 1], sharex="all"
)

trace = FitTrace(
    rectified_obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
subtracted = Background.two_sided(
    rectified_obj, trace, separation=60, width=20
).sub_image()
science = BoxcarExtract(subtracted, trace, width=11).spectrum
calibrated = result.solution.resample(science)

axs[0].plot(calibrated.spectral_axis.value, calibrated.flux.value, lw=0.6)
axs[0].set_ylabel(f"Flux [{calibrated.flux.unit.to_string('latex_inline')}]")

axs[1].axhline(0, c="0.6", lw=0.8)
axs[1].plot(axis, axis - prior_axis, lw=0.9, c="C1")
axs[1].set(
    xlim=(axis[0], axis[-1]),
    xlabel=r"Wavelength [$\AA$]",
    ylabel="automated $-$\ninteractive [$\\AA$]",
);

The top panel is the science spectrum on the automated solution's wavelength axis; the bottom is
the difference between the two chapters' solutions.

Between pixels 87 and 803 the two solutions agree to 0.27 Å, 0.05 pixels, well inside either
solution's rms. The difference grows toward the edges of the detector, to 1.04 Å at the red end,
where both fits are least constrained. It is not zero because the two chapters reach the same
`refine_fit(max_match_distance=2.0)` from different starting solutions, and each start hands the
2 Å window a slightly different set of lines at the margin, 26 here against Chapter 10's 25. The
two workflows are two ways into the same solution, to within the ambiguity the line list leaves.

## Summary

- Automated calibration is for **pipelines on stable, well-characterized instruments**. It starts
  from a prior drawn from checked calibrations of the same setup; here Chapter 10's stored solution
  stands in for them.
- **`fit_dispersion` searches inside bounds instead of starting from anchors**:
  `wavelength_bounds` and `dispersion_bounds` at the reference pixel and `higher_order_limits`, all
  bracketing the prior. It returns the same `WavelengthSolution1D`.
- **Always set `higher_order_limits` explicitly.** The automatic default can be tighter than the
  true coefficients, and then no seed can find the answer.
- **The bounds carry the prior.** At the bounds used here ten seeds all converge; widen any of
  them and convergence depends on the seed, with failures landing hundreds of Ångström away.
- Pass a `seed` to `fit_dispersion` to make a run reproducible.
- End with the interactive chapter's `refine_fit`. The automated solution then agrees with the
  interactive one to **0.05 pixels** where the lines constrain it, the difference being the
  ambiguity the line list leaves at the matching tolerance.
- A pipeline needs an **automated check that the fit converged**, which this book does not build;
  the matched line count and the fitted coverage are its natural inputs.

Next: [The 2D Wavelength Solution](12-wavelength-solution-2d.ipynb), which chains this solution
with the tilt solution into one object describing the whole detector.